
# Full Preliminary Multi-View Fusion Experiment

This notebook is for the **full curated multi-view building-damage dataset**
used in the MV-HarveyNET / Khajwal-style preliminary experiment.

It is stronger than a single pilot run because it:

- uses all usable buildings;
- keeps all views of one physical building in the same split;
- extracts frozen pretrained **ResNet18** features once;
- compares several fusion methods under the same conditions;
- repeats the experiment over multiple random seeds;
- reports mean and standard deviation;
- generates report-ready figures.

### Main comparison

\[
\text{Single View}
\rightarrow
\text{Mean Pooling}
\rightarrow
\text{CNN Fusion}
\rightarrow
\text{Attention Pooling}
\rightarrow
\text{GRU Fusion}
\rightarrow
\text{Cross-View Transformer}
\]

The purpose is to test whether learned cross-view interaction provides
additional benefit over simpler fusion. It does **not** assume in advance
that the Transformer must win.


In [ ]:

!pip -q install openpyxl scikit-learn pandas matplotlib pillow



## 1. Mount Google Drive and set dataset paths

You need:

- the folder containing the multi-view building images;
- the Excel/CSV file containing a building identifier and damage label.

The notebook attempts automatic ID/label detection. If it chooses incorrectly,
set `ID_COL` and `LABEL_COL` manually.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ===========================
# EDIT THE TWO PATHS ONLY
# ===========================
DATA_ROOT = "/content/drive/MyDrive/MV-HarveyNET/Final_images"
LABEL_FILE = "/content/drive/MyDrive/MV-HarveyNET/labels.xlsx"

# Actual spreadsheet columns
ID_COL = "Bldng ID"
LABEL_COL = "Building damage state"

MAX_VIEWS = 5
MIN_VIEWS = 4

# Repeated building-level splits
SEEDS = [11, 22, 33, 44, 55]

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

BATCH_SIZE = 32
MAX_EPOCHS = 35
PATIENCE = 6
LR = 2e-4
WEIGHT_DECAY = 1e-4

RESULT_ROOT = "/content/multiview_fusion_results"



## 2. Load annotations


In [ ]:

from pathlib import Path
import os, re, json, math, random, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision.models import resnet18, ResNet18_Weights

warnings.filterwarnings("ignore")

DATA_ROOT = Path(DATA_ROOT)
LABEL_FILE = Path(LABEL_FILE)
RESULT_ROOT = Path(RESULT_ROOT)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

if LABEL_FILE.suffix.lower() in [".xlsx", ".xls"]:
    labels_df = pd.read_excel(LABEL_FILE)
else:
    labels_df = pd.read_csv(LABEL_FILE)

print("Annotation shape:", labels_df.shape)
print("Columns:")
for i, c in enumerate(labels_df.columns):
    print(i, repr(c), "sample:", labels_df[c].dropna().astype(str).head(3).tolist())

display(labels_df.head())


## 3. Confirm the spreadsheet columns

The building labels come from:

- `Bldng ID` — building identifier
- `Building damage state` — target damage class

The image files have already been renamed into a much simpler form:

```text
View1_Building0.png
View2_Building0.png
...
View1_Building1.png
```

Therefore, the number after `Building` in each filename maps directly to
`Bldng ID`. The original Fulcrum photo UUIDs in the spreadsheet are not needed
for this processed image set.


In [ ]:
required_cols = [ID_COL, LABEL_COL]
missing = [c for c in required_cols if c not in labels_df.columns]

if missing:
    raise ValueError(
        f"Missing expected columns: {missing}\n"
        f"Available columns: {list(labels_df.columns)}"
    )

print("Building ID column :", ID_COL)
print("Damage label column:", LABEL_COL)

print("\nDamage-state counts in the spreadsheet:")
print(labels_df[LABEL_COL].value_counts(dropna=False).sort_index())

display(labels_df[[ID_COL, LABEL_COL]].head())


## 4. Match images using `ViewX_BuildingY`

The processed image filenames directly encode both the view number and building ID:

```text
View1_Building0.png
View2_Building0.png
View3_Building0.png
...
```

The integer `Y` after `Building` is matched directly to the spreadsheet
column `Bldng ID`. Views are sorted by their `ViewX` number.


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

all_images = sorted(
    p for p in DATA_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMG_EXTS
)

print("Images found:", len(all_images))

# Expected processed filename format:
# View1_Building0.png
# View5_Building123.jpg
NAME_RE = re.compile(
    r"^View(\d+)_Building(\d+)$",
    re.IGNORECASE
)

# building_id -> list of (view_number, path)
images_by_building = {}
unparsed_files = []

for p in all_images:
    m = NAME_RE.match(p.stem)
    if not m:
        unparsed_files.append(p)
        continue

    view_no = int(m.group(1))
    building_id = int(m.group(2))

    images_by_building.setdefault(building_id, []).append(
        (view_no, p)
    )

# Sort each building's views by View1, View2, ...
for bid in images_by_building:
    images_by_building[bid] = sorted(
        images_by_building[bid],
        key=lambda x: x[0]
    )

print("Building IDs detected in image filenames:", len(images_by_building))
print("Files not matching ViewX_BuildingY:", len(unparsed_files))

if unparsed_files:
    print("\nExamples of unparsed filenames:")
    for p in unparsed_files[:10]:
        print(" ", p.name)

# Construct one record per spreadsheet building.
records = []

for _, row in labels_df.iterrows():
    if pd.isna(row[ID_COL]) or pd.isna(row[LABEL_COL]):
        continue

    try:
        building_id = int(float(row[ID_COL]))
    except Exception:
        continue

    view_pairs = images_by_building.get(building_id, [])
    view_pairs = view_pairs[:MAX_VIEWS]

    records.append({
        "building_id": building_id,
        "label_raw": row[LABEL_COL],
        "n_views": len(view_pairs),
        "view_numbers": [v for v, _ in view_pairs],
        "image_paths": [str(p) for _, p in view_pairs],
    })

buildings_all = pd.DataFrame(records)

buildings = buildings_all[
    buildings_all["n_views"] >= MIN_VIEWS
].reset_index(drop=True)

print("\nBuildings in spreadsheet with non-missing labels:", len(buildings_all))
print(
    "Usable buildings with at least",
    MIN_VIEWS,
    "views:",
    len(buildings)
)

print("\nMatched view-count distribution:")
print(buildings_all["n_views"].value_counts().sort_index())

print("\nExample mappings:")
display(
    buildings_all[
        ["building_id", "label_raw", "n_views", "view_numbers", "image_paths"]
    ].head(10)
)

# Check which spreadsheet building IDs have no images.
missing_image_buildings = buildings_all.loc[
    buildings_all["n_views"] == 0,
    "building_id"
].tolist()

print("\nBuildings with no matched images:", len(missing_image_buildings))
print("First missing IDs:", missing_image_buildings[:20])

if len(buildings) == 0:
    raise RuntimeError(
        "No buildings were matched. Check DATA_ROOT and confirm the images "
        "are named like View1_Building0.png."
    )

# Save transparent mapping for reproducibility.
mapping_csv = RESULT_ROOT / "building_to_images_mapping.csv"
mapping_for_csv = buildings_all.copy()
mapping_for_csv["view_numbers"] = mapping_for_csv["view_numbers"].apply(
    lambda xs: ",".join(map(str, xs))
)
mapping_for_csv["image_paths"] = mapping_for_csv["image_paths"].apply(
    lambda xs: " | ".join(xs)
)
mapping_for_csv.to_csv(mapping_csv, index=False)

print("\nSaved mapping:", mapping_csv)


### Sanity check — verify the recovered multi-view grouping

You should now see views such as:

```text
View1_Building0.png
View2_Building0.png
...
```

grouped together with the damage state from spreadsheet row `Bldng ID = 0`.

With 2177 images and 436 spreadsheet buildings, most buildings are expected
to have approximately five processed views, with a small number possibly
having fewer or missing images. Check several examples visually before training.


In [ ]:

def show_buildings(df, n=4, seed=7):
    sample = df.sample(min(n, len(df)), random_state=seed)
    for _, row in sample.iterrows():
        paths = row["image_paths"]
        fig, axes = plt.subplots(1, len(paths), figsize=(3.2 * len(paths), 3))
        if len(paths) == 1:
            axes = [axes]
        for ax, p in zip(axes, paths):
            ax.imshow(Image.open(p).convert("RGB"))
            ax.axis("off")
            ax.set_title(Path(p).name, fontsize=8)
        fig.suptitle(
            f'Building {row["building_id"]} | label={row["label_raw"]}'
        )
        fig.tight_layout()
        plt.show()

show_buildings(buildings, n=4)



## 5. Encode labels and inspect class balance


In [ ]:

label_values = sorted(buildings["label_raw"].unique(), key=lambda x: str(x))
label_to_idx = {v: i for i, v in enumerate(label_values)}
idx_to_label = {i: v for v, i in label_to_idx.items()}

buildings["label"] = buildings["label_raw"].map(label_to_idx).astype(int)
NUM_CLASSES = len(label_values)

print("Label mapping:")
for k, v in label_to_idx.items():
    print(repr(k), "->", v)

print("\nClass counts:")
print(buildings["label_raw"].value_counts().sort_index())

class_counts = buildings["label_raw"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([str(x) for x in class_counts.index], class_counts.values)
ax.set_xlabel("Damage class")
ax.set_ylabel("Number of buildings")
ax.set_title("Building-Level Class Distribution")
ax.grid(axis="y", linestyle=":", alpha=0.5)
fig.tight_layout()
fig.savefig(
    RESULT_ROOT / "class_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()



## 6. Precompute frozen ResNet18 features

Each image is converted into a 512-dimensional feature vector once.
Repeated fusion experiments then train only small fusion modules.


In [ ]:

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

weights = ResNet18_Weights.DEFAULT
encoder = resnet18(weights=weights)
encoder.fc = nn.Identity()
encoder = encoder.to(DEVICE).eval()
preprocess = weights.transforms()

all_unique_paths = sorted(
    set(p for paths in buildings["image_paths"] for p in paths)
)
print("Unique images:", len(all_unique_paths))

cache_path = RESULT_ROOT / "resnet18_feature_cache.pt"

if cache_path.exists():
    feature_cache = torch.load(cache_path, map_location="cpu")
    print("Loaded cached features:", len(feature_cache))
else:
    feature_cache = {}
    with torch.no_grad():
        for i, path in enumerate(all_unique_paths):
            img = Image.open(path).convert("RGB")
            x = preprocess(img).unsqueeze(0).to(DEVICE)
            z = encoder(x).squeeze(0).cpu()
            feature_cache[path] = z
            if (i + 1) % 100 == 0 or (i + 1) == len(all_unique_paths):
                print(f"Encoded {i+1}/{len(all_unique_paths)}")
    torch.save(feature_cache, cache_path)

FEATURE_DIM = next(iter(feature_cache.values())).numel()
print("Feature dimension:", FEATURE_DIM)



## 7. Build padded building-level tensors


In [ ]:

N = len(buildings)

X = torch.zeros(N, MAX_VIEWS, FEATURE_DIM, dtype=torch.float32)
M = torch.zeros(N, MAX_VIEWS, dtype=torch.bool)
y = torch.tensor(buildings["label"].to_numpy(), dtype=torch.long)

for i, paths in enumerate(buildings["image_paths"]):
    for j, p in enumerate(paths[:MAX_VIEWS]):
        X[i, j] = feature_cache[p]
        M[i, j] = True

print("Feature tensor:", X.shape)
print("Mask tensor:", M.shape)
print("Labels:", y.shape)



## 8. Building-level stratified splitting


In [ ]:

def make_split(labels, seed):
    idx = np.arange(len(labels))

    train_idx, temp_idx = train_test_split(
        idx,
        test_size=(1.0 - TRAIN_FRAC),
        random_state=seed,
        stratify=labels
    )

    relative_test = TEST_FRAC / (VAL_FRAC + TEST_FRAC)

    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=relative_test,
        random_state=seed,
        stratify=labels[temp_idx]
    )

    return train_idx, val_idx, test_idx

tr, va, te = make_split(buildings["label"].to_numpy(), SEEDS[0])
print("Example split:", len(tr), len(va), len(te))



## 9. Fusion models


In [ ]:

class SingleViewClassifier(nn.Module):
    def __init__(self, d, c):
        super().__init__()
        self.classifier = nn.Linear(d, c)

    def forward(self, x, mask, return_embedding=False):
        first_idx = mask.float().argmax(dim=1)
        z = x[torch.arange(x.size(0), device=x.device), first_idx]
        logits = self.classifier(z)
        return (logits, z) if return_embedding else logits


class MeanPoolClassifier(nn.Module):
    def __init__(self, d, c):
        super().__init__()
        self.classifier = nn.Linear(d, c)

    def forward(self, x, mask, return_embedding=False):
        m = mask.unsqueeze(-1).float()
        z = (x * m).sum(dim=1) / m.sum(dim=1).clamp_min(1.0)
        logits = self.classifier(z)
        return (logits, z) if return_embedding else logits


class CNNFusionClassifier(nn.Module):
    def __init__(self, d, c, hidden=256):
        super().__init__()
        self.proj = nn.Linear(d, hidden)
        self.conv = nn.Sequential(
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(hidden, c)

    def forward(self, x, mask, return_embedding=False):
        h = self.proj(x)
        h = self.conv(h.transpose(1, 2)).transpose(1, 2)
        m = mask.unsqueeze(-1).float()
        z = (h * m).sum(dim=1) / m.sum(dim=1).clamp_min(1.0)
        logits = self.classifier(z)
        return (logits, z) if return_embedding else logits


class AttentionPoolClassifier(nn.Module):
    def __init__(self, d, c, hidden=256):
        super().__init__()
        self.proj = nn.Linear(d, hidden)
        self.score = nn.Linear(hidden, 1)
        self.classifier = nn.Linear(hidden, c)

    def forward(self, x, mask, return_embedding=False):
        h = torch.tanh(self.proj(x))
        a = self.score(h).squeeze(-1)
        a = a.masked_fill(~mask, -1e9)
        w = torch.softmax(a, dim=1)
        z = (h * w.unsqueeze(-1)).sum(dim=1)
        logits = self.classifier(z)
        return (logits, z) if return_embedding else logits


class GRUFusionClassifier(nn.Module):
    def __init__(self, d, c, hidden=256):
        super().__init__()
        self.proj = nn.Linear(d, hidden)
        self.gru = nn.GRU(hidden, hidden, batch_first=True)
        self.classifier = nn.Linear(hidden, c)

    def forward(self, x, mask, return_embedding=False):
        h = self.proj(x)
        out, _ = self.gru(h)
        lengths = mask.sum(dim=1).clamp_min(1)
        last = (lengths - 1).long()
        z = out[torch.arange(out.size(0), device=out.device), last]
        logits = self.classifier(z)
        return (logits, z) if return_embedding else logits


class CrossViewTransformer(nn.Module):
    def __init__(
        self, d, c, hidden=256, heads=4, layers=2, dropout=0.15
    ):
        super().__init__()
        self.proj = nn.Linear(d, hidden)
        self.building_token = nn.Parameter(torch.zeros(1, 1, hidden))
        nn.init.normal_(self.building_token, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=heads,
            dim_feedforward=hidden * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(hidden)
        self.classifier = nn.Linear(hidden, c)

    def forward(self, x, mask, return_embedding=False):
        h = self.proj(x)
        b = h.size(0)

        token = self.building_token.expand(b, -1, -1)
        seq = torch.cat([token, h], dim=1)

        token_mask = torch.ones(
            b, 1, dtype=torch.bool, device=x.device
        )
        valid = torch.cat([token_mask, mask], dim=1)

        out = self.encoder(
            seq,
            src_key_padding_mask=~valid
        )
        z = self.norm(out[:, 0])
        logits = self.classifier(z)

        return (logits, z) if return_embedding else logits


MODEL_FACTORIES = {
    "Single View": lambda: SingleViewClassifier(FEATURE_DIM, NUM_CLASSES),
    "Mean Pooling": lambda: MeanPoolClassifier(FEATURE_DIM, NUM_CLASSES),
    "CNN Fusion": lambda: CNNFusionClassifier(FEATURE_DIM, NUM_CLASSES),
    "Attention Pooling": lambda: AttentionPoolClassifier(FEATURE_DIM, NUM_CLASSES),
    "GRU Fusion": lambda: GRUFusionClassifier(FEATURE_DIM, NUM_CLASSES),
    "Cross-View Transformer": lambda: CrossViewTransformer(FEATURE_DIM, NUM_CLASSES),
}

print(list(MODEL_FACTORIES.keys()))



## 10. Training and evaluation helpers


In [ ]:

class FeatureDataset(Dataset):
    def __init__(self, X, M, y, indices):
        self.X = X[indices]
        self.M = M[indices]
        self.y = y[indices]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.M[i], self.y[i]


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict(model, loader, return_embeddings=False):
    model.eval()
    ys, preds, probs, embs = [], [], [], []

    for xb, mb, yb in loader:
        xb = xb.to(DEVICE)
        mb = mb.to(DEVICE)

        if return_embeddings:
            logits, z = model(xb, mb, return_embedding=True)
            embs.append(z.cpu().numpy())
        else:
            logits = model(xb, mb)

        p = torch.softmax(logits, dim=1)

        ys.append(yb.numpy())
        preds.append(p.argmax(dim=1).cpu().numpy())
        probs.append(p.cpu().numpy())

    output = (
        np.concatenate(ys),
        np.concatenate(preds),
        np.concatenate(probs),
    )

    if return_embeddings:
        return (*output, np.concatenate(embs))
    return output


def train_one(model, train_loader, val_loader, class_weights):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE)
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_f1 = -1.0
    no_improve = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []

        for xb, mb, yb in train_loader:
            xb = xb.to(DEVICE)
            mb = mb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb, mb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        yv, pv, _ = predict(model, val_loader)
        val_f1 = f1_score(
            yv, pv, average="macro", zero_division=0
        )
        val_acc = accuracy_score(yv, pv)

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "val_macro_f1": float(val_f1),
            "val_accuracy": float(val_acc),
        })

        if val_f1 > best_f1 + 1e-5:
            best_f1 = val_f1
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)



## 11. Run repeated experiments


In [ ]:

all_results = []
best_runs = {}
labels_np = buildings["label"].to_numpy()

for seed in SEEDS:
    print("\n" + "=" * 80)
    print("SEED:", seed)

    train_idx, val_idx, test_idx = make_split(labels_np, seed)

    train_loader = DataLoader(
        FeatureDataset(X, M, y, train_idx),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    val_loader = DataLoader(
        FeatureDataset(X, M, y, val_idx),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    test_loader = DataLoader(
        FeatureDataset(X, M, y, test_idx),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    train_counts = np.bincount(
        labels_np[train_idx],
        minlength=NUM_CLASSES
    )
    class_weights = len(train_idx) / (
        NUM_CLASSES * np.maximum(train_counts, 1)
    )
    class_weights = torch.tensor(
        class_weights, dtype=torch.float32
    )

    for model_name, factory in MODEL_FACTORIES.items():
        print("Training:", model_name)
        seed_everything(seed)

        model, history = train_one(
            factory(),
            train_loader,
            val_loader,
            class_weights,
        )

        yt, yp, prob, emb = predict(
            model,
            test_loader,
            return_embeddings=True,
        )

        acc = accuracy_score(yt, yp)
        mf1 = f1_score(
            yt, yp, average="macro", zero_division=0
        )

        all_results.append({
            "seed": seed,
            "model": model_name,
            "accuracy": acc,
            "macro_f1": mf1,
            "epochs": len(history),
            "n_train": len(train_idx),
            "n_val": len(val_idx),
            "n_test": len(test_idx),
        })

        # Stored only for later visualization.
        if (
            model_name not in best_runs
            or mf1 > best_runs[model_name]["macro_f1"]
        ):
            best_runs[model_name] = {
                "seed": seed,
                "accuracy": acc,
                "macro_f1": mf1,
                "model_state": {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                },
                "y_true": yt,
                "y_pred": yp,
                "prob": prob,
                "embeddings": emb,
                "test_idx": test_idx.copy(),
                "history": history.copy(),
            }

results = pd.DataFrame(all_results)
results.to_csv(
    RESULT_ROOT / "repeated_seed_results.csv",
    index=False
)

display(results)



## 12. Mean ± standard deviation across repeated splits


In [ ]:

summary = (
    results.groupby("model")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
    )
    .reset_index()
    .sort_values("macro_f1_mean", ascending=False)
)

summary.to_csv(
    RESULT_ROOT / "model_summary_mean_std.csv",
    index=False
)

display(summary)



## 13. Figure — fusion performance comparison


In [ ]:

plot_df = summary.copy()
xpos = np.arange(len(plot_df))
width = 0.36

fig, ax = plt.subplots(figsize=(12, 5.5))

ax.bar(
    xpos - width / 2,
    100 * plot_df["accuracy_mean"],
    width,
    yerr=100 * plot_df["accuracy_std"],
    capsize=4,
    label="Accuracy",
)

ax.bar(
    xpos + width / 2,
    100 * plot_df["macro_f1_mean"],
    width,
    yerr=100 * plot_df["macro_f1_std"],
    capsize=4,
    label="Macro-F1",
)

ax.set_xticks(xpos)
ax.set_xticklabels(
    plot_df["model"],
    rotation=25,
    ha="right",
)
ax.set_ylabel("Performance (%)")
ax.set_title(
    "Preliminary Multi-View Fusion Performance Across Repeated Splits"
)
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)

fig.tight_layout()

path = RESULT_ROOT / "fusion_performance_mean_std.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", path)



## 14. Figure — Transformer confusion matrix


In [ ]:

run = best_runs["Cross-View Transformer"]

cm = confusion_matrix(
    run["y_true"],
    run["y_pred"],
    labels=np.arange(NUM_CLASSES),
)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm)
fig.colorbar(im, ax=ax)

tick_labels = [
    str(idx_to_label[i])
    for i in range(NUM_CLASSES)
]
ax.set_xticks(np.arange(NUM_CLASSES))
ax.set_yticks(np.arange(NUM_CLASSES))
ax.set_xticklabels(
    tick_labels,
    rotation=45,
    ha="right",
)
ax.set_yticklabels(tick_labels)

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center", va="center"
        )

ax.set_xlabel("Predicted damage class")
ax.set_ylabel("True damage class")
ax.set_title(
    "Cross-View Transformer Confusion Matrix\n"
    f'Seed {run["seed"]} | '
    f'Accuracy={100 * run["accuracy"]:.1f}% | '
    f'Macro-F1={run["macro_f1"]:.3f}'
)

fig.tight_layout()

path = RESULT_ROOT / "transformer_confusion_matrix.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
plt.show()



## 15. Figure — per-class F1 for three key comparisons


In [ ]:

compare_models = [
    "Single View",
    "Mean Pooling",
    "Cross-View Transformer",
]

rows = []

for model_name in compare_models:
    r = best_runs[model_name]
    report = classification_report(
        r["y_true"],
        r["y_pred"],
        labels=np.arange(NUM_CLASSES),
        output_dict=True,
        zero_division=0,
    )

    for c in range(NUM_CLASSES):
        rows.append({
            "model": model_name,
            "class": str(idx_to_label[c]),
            "f1": report[str(c)]["f1-score"],
        })

per_class_df = pd.DataFrame(rows)
per_class_df.to_csv(
    RESULT_ROOT / "per_class_f1_selected_models.csv",
    index=False
)

classes = [
    str(idx_to_label[c])
    for c in range(NUM_CLASSES)
]
xpos = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5.2))

for offset, model_name in zip(
    [-width, 0, width],
    compare_models
):
    vals = (
        per_class_df[
            per_class_df["model"] == model_name
        ]
        .set_index("class")
        .loc[classes]["f1"]
        .to_numpy()
    )

    ax.bar(
        xpos + offset,
        vals,
        width,
        label=model_name,
    )

ax.set_xticks(xpos)
ax.set_xticklabels(classes)
ax.set_xlabel("Damage class")
ax.set_ylabel("F1 score")
ax.set_title("Per-Class Preliminary Performance")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)

fig.tight_layout()

path = RESULT_ROOT / "per_class_f1_comparison.png"
fig.savefig(path, dpi=300, bbox_inches="tight")
plt.show()



## 16. Preliminary robustness test — fewer available views

The strongest Transformer run is evaluated after progressively reducing
the number of valid views. This is a **stress test**, not a separately
retrained model for each value of \(K\).


In [ ]:

best_seed = run["seed"]
_, _, test_idx = make_split(labels_np, best_seed)

Xtest = X[test_idx].clone()
Mtest = M[test_idx].clone()
ytest = y[test_idx].clone()

model = MODEL_FACTORIES[
    "Cross-View Transformer"
]().to(DEVICE)

model.load_state_dict(run["model_state"])
model.eval()

robustness_rows = []

for k_keep in range(1, MAX_VIEWS + 1):
    Xk = Xtest.clone()
    Mk = Mtest.clone()

    for i in range(len(Mk)):
        valid = torch.where(Mk[i])[0]
        if len(valid) > k_keep:
            drop = valid[k_keep:]
            Mk[i, drop] = False
            Xk[i, drop] = 0

    loader = DataLoader(
        TensorDataset(Xk, Mk, ytest),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    yt, yp, _ = predict(model, loader)

    robustness_rows.append({
        "views_available": k_keep,
        "accuracy": accuracy_score(yt, yp),
        "macro_f1": f1_score(
            yt, yp,
            average="macro",
            zero_division=0,
        ),
    })

robustness = pd.DataFrame(robustness_rows)
robustness.to_csv(
    RESULT_ROOT /
    "transformer_missing_view_robustness.csv",
    index=False
)

display(robustness)

fig, ax = plt.subplots(figsize=(7.5, 4.8))

ax.plot(
    robustness["views_available"],
    100 * robustness["accuracy"],
    marker="o",
    label="Accuracy",
)
ax.plot(
    robustness["views_available"],
    100 * robustness["macro_f1"],
    marker="o",
    label="Macro-F1",
)

ax.set_xticks(range(1, MAX_VIEWS + 1))
ax.set_xlabel("Maximum available views")
ax.set_ylabel("Performance (%)")
ax.set_title(
    "Preliminary Transformer Robustness to Fewer Views"
)
ax.legend()
ax.grid(linestyle=":", alpha=0.5)

fig.tight_layout()

path = (
    RESULT_ROOT /
    "transformer_missing_view_robustness.png"
)
fig.savefig(path, dpi=300, bbox_inches="tight")
plt.show()



## 17. PCA visualization of learned building representations

This is an exploratory report visualization. It should not be interpreted
as proof of class separability.


In [ ]:

emb = run["embeddings"]
yt = run["y_true"]

pca = PCA(n_components=2)
coords = pca.fit_transform(emb)

fig, ax = plt.subplots(figsize=(7, 6))

for c in range(NUM_CLASSES):
    sel = yt == c
    ax.scatter(
        coords[sel, 0],
        coords[sel, 1],
        label=str(idx_to_label[c]),
        alpha=0.75,
    )

ax.set_xlabel("PCA component 1")
ax.set_ylabel("PCA component 2")
ax.set_title(
    "Transformer Building-Level Representations (PCA)"
)
ax.legend(title="Damage class")
ax.grid(linestyle=":", alpha=0.4)

fig.tight_layout()

path = (
    RESULT_ROOT /
    "transformer_building_embeddings_pca.png"
)
fig.savefig(path, dpi=300, bbox_inches="tight")
plt.show()

print(
    "Explained variance of two components:",
    pca.explained_variance_ratio_.sum()
)



## 18. Automatically create a concise summary


In [ ]:

single = summary[
    summary["model"] == "Single View"
].iloc[0]

meanp = summary[
    summary["model"] == "Mean Pooling"
].iloc[0]

trans = summary[
    summary["model"] == "Cross-View Transformer"
].iloc[0]

lines = [
    "PRELIMINARY MULTI-VIEW FUSION RESULTS",
    "",
    f"Usable buildings: {len(buildings)}",
    f"Damage classes: {NUM_CLASSES}",
    f"Repeated seeds: {len(SEEDS)}",
    (
        "Train/validation/test fractions: "
        f"{TRAIN_FRAC:.2f}/{VAL_FRAC:.2f}/{TEST_FRAC:.2f}"
    ),
    "",
    "Single View:",
    (
        f"  Accuracy = {100 * single.accuracy_mean:.2f}% "
        f"± {100 * single.accuracy_std:.2f}"
    ),
    (
        f"  Macro-F1 = {single.macro_f1_mean:.4f} "
        f"± {single.macro_f1_std:.4f}"
    ),
    "",
    "Mean Pooling:",
    (
        f"  Accuracy = {100 * meanp.accuracy_mean:.2f}% "
        f"± {100 * meanp.accuracy_std:.2f}"
    ),
    (
        f"  Macro-F1 = {meanp.macro_f1_mean:.4f} "
        f"± {meanp.macro_f1_std:.4f}"
    ),
    "",
    "Cross-View Transformer:",
    (
        f"  Accuracy = {100 * trans.accuracy_mean:.2f}% "
        f"± {100 * trans.accuracy_std:.2f}"
    ),
    (
        f"  Macro-F1 = {trans.macro_f1_mean:.4f} "
        f"± {trans.macro_f1_std:.4f}"
    ),
    "",
    "Transformer minus Single View:",
    (
        "  Accuracy difference = "
        f"{100 * (trans.accuracy_mean - single.accuracy_mean):+.2f} "
        "percentage points"
    ),
    (
        "  Macro-F1 difference = "
        f"{trans.macro_f1_mean - single.macro_f1_mean:+.4f}"
    ),
    "",
    "Transformer minus Mean Pooling:",
    (
        "  Accuracy difference = "
        f"{100 * (trans.accuracy_mean - meanp.accuracy_mean):+.2f} "
        "percentage points"
    ),
    (
        "  Macro-F1 difference = "
        f"{trans.macro_f1_mean - meanp.macro_f1_mean:+.4f}"
    ),
    "",
    (
        "Interpretation: these are preliminary fusion results on a "
        "curated multi-view image dataset. They do not yet validate "
        "the final continuous-video selection pipeline."
    ),
]

summary_text = "\n".join(lines)
print(summary_text)

(
    RESULT_ROOT /
    "preliminary_results_summary.txt"
).write_text(summary_text)



## 19. Create a LaTeX-ready table


In [ ]:

table_rows = []

for _, r in summary.sort_values(
    "macro_f1_mean",
    ascending=False
).iterrows():
    row = (
        f'{r["model"]} & '
        f'{100 * r["accuracy_mean"]:.2f} '
        f'$\\pm$ {100 * r["accuracy_std"]:.2f} & '
        f'{r["macro_f1_mean"]:.4f} '
        f'$\\pm$ {r["macro_f1_std"]:.4f} \\\\'
    )
    table_rows.append(row)

latex_table = "\n".join([
    r"\begin{table}[htbp]",
    r"\centering",
    (
        r"\caption{Preliminary multi-view fusion performance "
        r"across repeated building-level splits.}"
    ),
    r"\begin{tabular}{lcc}",
    r"\hline",
    r"Method & Accuracy (\%) & Macro-F1 \\",
    r"\hline",
    *table_rows,
    r"\hline",
    r"\end{tabular}",
    r"\label{tab:preliminary_fusion}",
    r"\end{table}",
])

print(latex_table)

(
    RESULT_ROOT /
    "preliminary_results_table.tex"
).write_text(latex_table)



## 20. Download all results


In [ ]:

zip_path = shutil.make_archive(
    "/content/multiview_fusion_results",
    "zip",
    RESULT_ROOT,
)

print("Created:", zip_path)

from google.colab import files
files.download(zip_path)



# Send me these results afterward

The easiest option is to upload:

**`multiview_fusion_results.zip`**

The most useful files inside are:

- `model_summary_mean_std.csv`
- `repeated_seed_results.csv`
- `fusion_performance_mean_std.png`
- `transformer_confusion_matrix.png`
- `per_class_f1_comparison.png`
- `transformer_missing_view_robustness.png`
- `transformer_building_embeddings_pca.png`
- `class_distribution.png`
- `preliminary_results_summary.txt`

Then I can select the strongest 2–3 visualizations, interpret the results,
and write the final **Preliminary Results** section in LaTeX.


# NEW EXPERIMENT — Does quality + visibility + diversity selection help the Transformer?

This extension uses **your exact existing data mapping, pretrained ResNet18 features, Transformer architecture, five stratified building-level splits, hyperparameters, and early stopping**.

The original experiment trained the Transformer using every available curated view (normally 4–5). Here we retrain the **same architecture from scratch** using **K=3** selected views per building. We compare:

| Input to Transformer | Number of views | What it tests |
|---|---:|---|
| All available views | 4–5 | Existing unselected benchmark |
| Uniformly spaced views | 3 | Simple subset baseline |
| Random views | 3 | Arbitrary selection |
| Quality + visibility | 3 | Usability alone |
| Diversity only | 3 | Complementarity alone |
| **Quality + visibility + diversity** | **3** | **Proposed selection rule** |

**Important scientific limitations**

- MV-HarveyNET already contains **cropped, curated photographs**, not continuous video. The results test *selection among available physical views*, not the full continuous-video selection pipeline.
- The **building-visibility** score here is a *pretrained semantic segmentation proxy* (predicted fraction of pixels labeled as building/house). It is not a verified ground-truth building mask; we will display a diagnostic to check whether it behaves meaningfully on damaged buildings.
- Selection uses **no damage class labels**, so there is no damage-label leakage into frame selection.
- Reducing 4–5 photos to 3 might improve, preserve, or reduce accuracy. We will report what actually happens.

**How to run:** Run the existing notebook first through its model/data/training-function cells (sections 1–10) so that `X`, `M`, `y`, `buildings`, `encoder`, `make_split`, `train_one`, etc., exist. Then run the added cells below. You may skip the original repeated-method experiment if you only want this selection experiment; the extension independently trains its all-view Transformer baseline.


In [ ]:
# NEW 1. Settings for the view-selection experiment
!pip -q install 'transformers>=4.40,<5' opencv-python-headless

from pathlib import Path
import cv2
from IPython.display import display

SELECTION_K = 3
SELECTION_SEEDS = list(SEEDS)   # same five split seeds as your original results
SELECTION_ALPHA = 1/3         # normalized sharpness
SELECTION_BETA = 1/3          # exposure
SELECTION_GAMMA = 1/3         # building segmentation visibility proxy
QUALITY_DIVERSITY_WEIGHT = 0.50
QUALITY_FILTER_FRACTION = 0.20  # screen lowest 20% by quality, if K views remain

EXPERIMENT_DIR = RESULT_ROOT / 'view_selection_transformer_experiment'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

required_vars = ['X','M','y','buildings','SEEDS','NUM_CLASSES','FEATURE_DIM',
                 'train_one','predict','make_split','CrossViewTransformer', 'DEVICE']
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError('Please run the original notebook through sections 1–10 first. Missing: '
                       + ', '.join(missing_vars))
assert int(M.sum(dim=1).min()) >= SELECTION_K, 'Some buildings have fewer than K views.'
print('Buildings:',len(buildings),' K:',SELECTION_K,' device:',DEVICE)
print('Experimental results:',EXPERIMENT_DIR)


## NEW 2. Calculate and cache the three per-view scores

Each view is scored using:

- **Sharpness:** variance of the Laplacian (computed after consistent resizing).
- **Exposure:** fraction of pixels not too dark or overexposed.
- **Visibility:** proportion of pixels labeled as building-like by **SegFormer-B0/ADE20K**. This is **an approximate visibility proxy**, not a dedicated damaged-building detector. In particular, roof rubble, unusual facades, and close-up photos may cause poor semantic segmentation.

The scores are cached in a separate CSV in the experiment directory, so subsequent runs should not need to recompute them. For segmentation we recommend **Colab T4 GPU**. The first run will download a pretrained model and can take several minutes.


In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

SEGMENTER_ID = 'nvidia/segformer-b0-finetuned-ade-512-512'
score_csv = EXPERIMENT_DIR / 'image_quality_visibility_scores.csv'
all_paths_for_selection = sorted({p for arr in buildings['image_paths'] for p in arr})

score_df = pd.read_csv(score_csv) if score_csv.exists() else pd.DataFrame()
if not score_df.empty and 'image_path' in score_df.columns:
    valid_scores = set(score_df['image_path'].astype(str))
    already_scored = score_df.set_index('image_path').to_dict('index')
else:
    valid_scores, already_scored = set(), {}

missing_paths = [p for p in all_paths_for_selection if p not in valid_scores]
print(f'Already cached: {len(all_paths_for_selection)-len(missing_paths)} / {len(all_paths_for_selection)}')

if missing_paths:
    processor = SegformerImageProcessor.from_pretrained(SEGMENTER_ID)
    segmodel = SegformerForSemanticSegmentation.from_pretrained(SEGMENTER_ID).to(DEVICE).eval()
    id_to_name = {int(i): str(name).lower() for i, name in segmodel.config.id2label.items()}
    BUILDING_CLASS_IDS = sorted([
        i for i, name in id_to_name.items()
        if any(token in name for token in ('building', 'house', 'skyscraper'))
    ])
    print('Segmentation categories counted as buildings:',
          [(i,id_to_name[i]) for i in BUILDING_CLASS_IDS])
    if not BUILDING_CLASS_IDS:
        raise RuntimeError('The segmentation model exposed no building class. Cannot claim visibility scores.')

    @torch.no_grad()
    def score_image(path):
        # Normalize spatial scale for image-quality measurements.
        img = Image.open(path).convert('RGB')
        small = np.asarray(img.resize((512,512)))
        gray = cv2.cvtColor(small,cv2.COLOR_RGB2GRAY)
        lap = float(cv2.Laplacian(gray,cv2.CV_64F).var())
        exposure = float(1.0 - np.mean((gray < 25) | (gray > 240)))
        inputs = processor(images=img, return_tensors='pt').to(DEVICE)
        logits = segmodel(**inputs).logits
        segmentation = logits.argmax(dim=1).squeeze().cpu().numpy()
        visibility = float(np.isin(segmentation,BUILDING_CLASS_IDS).mean())
        return {'sharpness_raw':lap,'exposure':exposure,'visibility_proxy':visibility}

    for index, path in enumerate(missing_paths,1):
        try:
            already_scored[path] = score_image(path)
        except Exception as err:
            raise RuntimeError(f'Could not score {path}: {err}') from err
        if index % 100 == 0 or index == len(missing_paths):
            print('Scored new images:',index,'/',len(missing_paths))
            # Write incremental cache: recover from Colab interruptions.
            part = pd.DataFrame([dict(image_path=p,**already_scored[p]) for p in already_scored])
            part.to_csv(score_csv,index=False)

score_df = pd.DataFrame([{'image_path':p, **already_scored[p]} for p in all_paths_for_selection])
score_df.to_csv(score_csv,index=False)
score_lookup = score_df.set_index('image_path')
print('\nQuality/visibility score summary:')
display(score_df[['sharpness_raw','exposure','visibility_proxy']].describe().round(3))
frac_zero = float((score_df['visibility_proxy'] < 0.01).mean())
print(f'Images with almost no detected building pixels: {frac_zero*100:.1f}%')
if frac_zero > 0.50:
    print('WARNING: the building visibility proxy is often nearly zero. Interpret it cautiously;')
    print('consider using a dedicated building segmenter in the final thesis.')
# Release segmentation memory before fusion training.
if 'segmodel' in globals():
    del segmodel
    if torch.cuda.is_available(): torch.cuda.empty_cache()


## NEW 3. Select K views for each building **without using damage labels**

To avoid numerical scale mismatch, sharpness and segmentation visibility are normalized **within each building's available views**. Exposure already lies between 0 and 1. The quality-visibility score is:

\[
Q = \alpha\,Q_{\rm sharp}+\beta\,Q_{\rm exposure}+\gamma\,Q_{\rm visibility}, \quad \alpha+\beta+\gamma=1.
\]

For the proposed selection, we start from the highest-quality view, then greedily balance the candidate's score against its **minimum cosine distance** to already selected ResNet18 feature embeddings. An initial low-quality candidate can be screened out when sufficient views remain. All selection strategies output the same number of views, except the full-view reference.


In [ ]:
def minmax_within_building(values):
    v = np.asarray(values,dtype=np.float64)
    lo, hi = float(np.min(v)),float(np.max(v))
    # Constant score has no relative ranking information.
    return np.full(len(v), 0.5) if hi-lo < 1e-10 else (v-lo)/(hi-lo)


def building_view_scores(building_row_index):
    paths = list(buildings.iloc[building_row_index]['image_paths'])
    valid = np.flatnonzero(M[building_row_index].numpy())
    assert len(paths) == len(valid), 'Image-path and feature-mask ordering do not agree.'
    frame_scores = score_lookup.loc[paths]
    sharp = minmax_within_building(np.log1p(frame_scores['sharpness_raw'].to_numpy()))
    exposure = np.clip(frame_scores['exposure'].to_numpy(dtype=float),0,1)
    visibility = minmax_within_building(frame_scores['visibility_proxy'].to_numpy(dtype=float))
    Q = (SELECTION_ALPHA*sharp + SELECTION_BETA*exposure + SELECTION_GAMMA*visibility)
    features = X[building_row_index,valid].numpy().astype(np.float64)
    features = features / np.maximum(np.linalg.norm(features,axis=1,keepdims=True),1e-9)
    similarity = np.clip(features@features.T,-1,1)
    return valid,Q,similarity,paths


def select_for_building(building_row_index, method, K=3, seed=11):
    valid,Q,sim,paths = building_view_scores(building_row_index)
    n=len(valid)
    K=min(K,n)
    indices=np.arange(n)

    if method == 'All available views':
        return valid.tolist()
    if method == 'Uniform K=3':
        order=np.unique(np.rint(np.linspace(0,n-1,K)).astype(int))
        return sorted(valid[order].tolist())
    if method == 'Random K=3':
        # Reproducible for each building and experiment seed.
        rng=np.random.default_rng(seed*1000003+building_row_index)
        return sorted(valid[rng.choice(indices,size=K,replace=False)].tolist())
    if method == 'Quality+visibility K=3':
        order=sorted(indices,key=lambda i:(-Q[i],i))[:K]
        return sorted(valid[order].tolist())

    # Greedy farthest-first diversity, optionally with quality/visibility.
    candidates=list(indices)
    if method == 'Quality+visibility+diversity K=3' and n>K:
        n_remove=min(int(np.floor(n*QUALITY_FILTER_FRACTION)),n-K)
        if n_remove>0:
            bad=set(sorted(indices,key=lambda i:(Q[i],i))[:n_remove])
            candidates=[i for i in candidates if i not in bad]

    # Normalize distances by maximum observed distance in THIS building.
    # This prevents small absolute cosine distances being swamped by Q.
    distances=np.clip(1-sim,0,2)
    max_distance=max(float(distances[np.ix_(candidates,candidates)].max()),1e-8)

    if method == 'Diversity only K=3':
        # Pick the point most different on average from the other views.
        selected=[max(candidates,key=lambda i:(float(np.mean(distances[i,candidates])), -i))]
        quality_weight=0.0
    elif method == 'Quality+visibility+diversity K=3':
        selected=[max(candidates,key=lambda i:(float(Q[i]),-i))]
        quality_weight=QUALITY_DIVERSITY_WEIGHT
    else:
        raise ValueError(f'Unknown selection strategy: {method}')

    while len(selected)<K:
        remaining=[i for i in candidates if i not in selected]
        next_index=max(
            remaining,
            key=lambda i:(
                quality_weight*float(Q[i]) +
                (1-quality_weight)*float(np.min(distances[i,selected]))/max_distance,
                -i,
            )
        )
        selected.append(next_index)
    return sorted(valid[selected].tolist())


# Make a new masked tensor with exactly the selected original view positions.
def make_selected_inputs(method,seed,K=SELECTION_K):
    X_new=torch.zeros_like(X)
    M_new=torch.zeros_like(M)
    selection_audit=[]
    for building_index in range(len(buildings)):
        chosen=select_for_building(building_index,method,K,seed)
        X_new[building_index,chosen]=X[building_index,chosen]
        M_new[building_index,chosen]=True
        selection_audit.append({
            'seed':seed,
            'method':method,
            'building_id':buildings.iloc[building_index]['building_id'],
            'selected_indices':','.join(map(str,chosen)),
            'selected_view_numbers':','.join(
                map(str,[buildings.iloc[building_index]['view_numbers'][int(j)] for j in chosen])
            ),
        })
    return X_new,M_new,selection_audit

METHODS = [
    'All available views',
    'Uniform K=3',
    'Random K=3',
    'Quality+visibility K=3',
    'Diversity only K=3',
    'Quality+visibility+diversity K=3',
]

# Sanity check and example selections.
for method in METHODS:
    X_example,M_example,rows = make_selected_inputs(method,SELECTION_SEEDS[0])
    expected=SELECTION_K if method!='All available views' else None
    if expected is not None:
        assert torch.all(M_example.sum(1)==expected),f'{method}: not exactly K per building'
    assert torch.all(M_example <= M),f'{method}: selected a non-existent view'
    print(f'{method:<39} | first building views: {rows[0]["selected_view_numbers"]}')
print('All view-selection sanity checks passed.')


## NEW 4. Train and evaluate the same cross-view Transformer on each selection rule

This trains one Transformer for each condition and split, always using:

- the same model architecture and ResNet18 features,
- the same building-level training/validation/test indices,
- the same number of selected views **K=3** for the selection-method comparison,
- identical hyperparameters and validation-based early stopping,
- **no test-label information in view selection**.

The reference condition uses all available 4–5 views. We recompute it in the same run so that the comparison is self-contained, even if your earlier results CSV is missing from Colab.


In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score,f1_score

selection_results=[]
selection_audit_all=[]
labels_for_split=buildings['label'].to_numpy()

for seed in SELECTION_SEEDS:
    train_idx,val_idx,test_idx=make_split(labels_for_split,seed)
    n_per_class=np.bincount(labels_for_split[train_idx],minlength=NUM_CLASSES)
    class_weight_values = len(train_idx)/(NUM_CLASSES*np.maximum(n_per_class,1))
    class_weights=torch.tensor(class_weight_values,dtype=torch.float32)

    for method in METHODS:
        print('\nSeed:',seed,' | method:',method)
        seed_everything(seed)
        X_sel,M_sel,audit=make_selected_inputs(method,seed=seed)
        selection_audit_all.extend(audit)

        train_loader=DataLoader(
            FeatureDataset(X_sel,M_sel,y,train_idx),batch_size=BATCH_SIZE,shuffle=True)
        valid_loader=DataLoader(
            FeatureDataset(X_sel,M_sel,y,val_idx),batch_size=BATCH_SIZE,shuffle=False)
        test_loader=DataLoader(
            FeatureDataset(X_sel,M_sel,y,test_idx),batch_size=BATCH_SIZE,shuffle=False)

        # Same architecture, same initialization seed, same optimization routine.
        model=CrossViewTransformer(FEATURE_DIM,NUM_CLASSES)
        model,history=train_one(model,train_loader,valid_loader,class_weights)
        yt,yp,_=predict(model,test_loader)
        acc=accuracy_score(yt,yp)
        macro=f1_score(yt,yp,average='macro',zero_division=0)
        selection_results.append({
            'seed':seed,'method':method,'K': 'all' if method=='All available views' else SELECTION_K,
            'n_train':len(train_idx),'n_val':len(val_idx),'n_test':len(test_idx),
            'epochs':len(history),'accuracy':float(acc),'macro_f1':float(macro),
        })
        print(f'Accuracy={100*acc:.2f}% | Macro-F1={macro:.4f}')
        # Incremental saving so partial Colab runs are recoverable.
        pd.DataFrame(selection_results).to_csv(EXPERIMENT_DIR/'per_seed_selection_results.csv',index=False)

selection_df=pd.DataFrame(selection_results)
pd.DataFrame(selection_audit_all).to_csv(EXPERIMENT_DIR/'selected_view_audit.csv',index=False)
print('\nResults for all conditions:')
display(selection_df)


## NEW 5. The only main figure and table needed for your proposal

We report **mean ± standard deviation over the five splits** and compare the proposed selector against uniform/random selection at the **same view budget**. A small accuracy difference with five test splits is not automatically a statistically reliable improvement.


In [ ]:
selection_summary = (
    selection_df.groupby('method',sort=False)
    .agg(accuracy_mean=('accuracy','mean'),accuracy_std=('accuracy','std'),
         macro_f1_mean=('macro_f1','mean'),macro_f1_std=('macro_f1','std'))
    .reindex(METHODS)
    .reset_index()
)
selection_summary.to_csv(EXPERIMENT_DIR/'view_selection_model_summary.csv',index=False)
display(selection_summary.round(4))

reference=selection_summary.set_index('method').loc['All available views']
proposed=selection_summary.set_index('method').loc['Quality+visibility+diversity K=3']
uniform=selection_summary.set_index('method').loc['Uniform K=3']
print('\nProposed (K=3) vs uniform (K=3):')
print(f'  Accuracy: {100*(proposed.accuracy_mean-uniform.accuracy_mean):+.2f} percentage points')
print(f'  Macro-F1: {proposed.macro_f1_mean-uniform.macro_f1_mean:+.4f}')
print('Proposed (K=3) vs all available views (4–5):')
print(f'  Accuracy: {100*(proposed.accuracy_mean-reference.accuracy_mean):+.2f} percentage points')
print(f'  Macro-F1: {proposed.macro_f1_mean-reference.macro_f1_mean:+.4f}')

# One restrained report-ready comparison figure.
fig,ax=plt.subplots(figsize=(12,5.2))
xs=np.arange(len(selection_summary)); bw=0.35
ax.bar(xs-bw/2,100*selection_summary['accuracy_mean'],bw,
       yerr=100*selection_summary['accuracy_std'],capsize=3,label='Accuracy')
ax.bar(xs+bw/2,100*selection_summary['macro_f1_mean'],bw,
       yerr=100*selection_summary['macro_f1_std'],capsize=3,label='Macro-F1')
ax.set_xticks(xs)
ax.set_xticklabels([
    'All views\n(4–5)','Uniform\n(K=3)','Random\n(K=3)',
    'Quality + visibility\n(K=3)','Diversity only\n(K=3)',
    'Quality + visibility\n+ diversity (K=3)'])
ax.set_ylabel('Performance (%)')
ax.set_title('Effect of representative-view selection on cross-view Transformer performance')
ax.set_ylim(0,100)
ax.grid(axis='y',linestyle=':',alpha=.4)
ax.legend()
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR/'view_selection_transformer_comparison.png',dpi=300,bbox_inches='tight')
plt.show()

# Generate a ready-to-edit LaTeX results table.
tex_lines=[r'\begin{table}[htbp]',r'\centering',
 r'\caption{Preliminary effect of representative-view selection on cross-view Transformer performance.}',
 r'\label{tab:preliminary_selection}',r'\begin{tabular}{lcc}',r'\hline',
 r'Method & Accuracy (\%) & Macro-F1 \\',r'\hline']
for _,r in selection_summary.iterrows():
    name=r['method'].replace('+',' + ')
    tex_lines.append(
        f'{name} & ${100*r.accuracy_mean:.2f} \\pm {100*r.accuracy_std:.2f}$ & '
        f'${r.macro_f1_mean:.4f} \\pm {r.macro_f1_std:.4f}$ ' + r'\\'
    )
tex_lines.extend([r'\hline',r'\end{tabular}',r'\end{table}'])
tex_path=EXPERIMENT_DIR/'view_selection_results_table.tex'
tex_path.write_text('\n'.join(tex_lines))
print('Saved LaTeX table to:',tex_path)


## NEW 6. Save the results you should send back

The ZIP contains one comparison figure, the summary and per-seed metrics, the image-scoring CSV, and a record of which views were selected for each building. You can upload it here for interpretation and inclusion in your thesis proposal.


In [ ]:
from google.colab import files
archive=shutil.make_archive('/content/view_selection_transformer_results','zip',EXPERIMENT_DIR)
print('Results ZIP:',archive)
files.download(archive)
